# Chapter 10 - Multiclass Classification

So far we have focused on binary classification: two classes, one decision boundary.
But many problems involve three or more classes — identifying handwritten digits (0-9),
classifying flower species, or predicting customer segments.

Logistic regression extends naturally to the multiclass setting through two strategies:
decomposing the problem into binary subproblems (One-vs-Rest, One-vs-One), or
generalizing the model directly via the softmax function.

**Topics covered:**
- One-vs-Rest (OvR) strategy
- One-vs-One (OvO) strategy
- Softmax regression (multinomial logistic regression)
- Iris dataset classification example
- Multiclass confusion matrix and classification_report
- Visualizing decision boundaries for 3+ classes
- When to use logistic regression vs other classifiers

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_iris, make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
)
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True

## One-vs-Rest (OvR) Strategy

The simplest approach: for $K$ classes, train **$K$ binary classifiers**. Each classifier
learns to separate one class from all the others.

For a 3-class problem (A, B, C):
- Classifier 1: A vs (B + C)
- Classifier 2: B vs (A + C)
- Classifier 3: C vs (A + B)

At prediction time, run all classifiers and pick the one with the **highest confidence**
(highest predicted probability).

**Pros:** Simple, works with any binary classifier, scales linearly with $K$.

**Cons:** Each classifier sees an imbalanced problem (1 class vs all others); classifiers
are trained independently so their probability scales may not be comparable.

In [ ]:
# Load the Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
target_names = iris.target_names
feature_names = iris.feature_names

print(f'Dataset shape: {X_iris.shape}')
print(f'Classes: {target_names}')
print(f'Class distribution: {np.bincount(y_iris)}')
print(f'Features: {feature_names}')

In [ ]:
# Train-test split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train_scaled.shape[0]}, Test: {X_test_scaled.shape[0]}')

In [ ]:
# One-vs-Rest (OvR)
# In sklearn, LogisticRegression uses OvR by default for multi_class='auto' with
# solvers that don't support multinomial
model_ovr = LogisticRegression(
    multi_class='ovr',
    solver='lbfgs',
    random_state=42,
    max_iter=1000,
)
model_ovr.fit(X_train_scaled, y_train)

y_pred_ovr = model_ovr.predict(X_test_scaled)
print(f'OvR Accuracy: {accuracy_score(y_test, y_pred_ovr):.4f}')
print()

# The model has K sets of coefficients (one per class)
print(f'Coefficient matrix shape: {model_ovr.coef_.shape}  (K classes x n features)')
print(f'Intercept shape: {model_ovr.intercept_.shape}')

## One-vs-One (OvO) Strategy

For $K$ classes, train a classifier for **every pair** of classes:
$\binom{K}{2} = \frac{K(K-1)}{2}$ classifiers.

For a 3-class problem:
- Classifier 1: A vs B
- Classifier 2: A vs C
- Classifier 3: B vs C

At prediction time, each classifier "votes" for its winner, and the class with the
**most votes** wins.

**Pros:** Each classifier trains on a smaller, balanced subset; can work better for
algorithms that struggle with large datasets (e.g., SVM).

**Cons:** Number of classifiers grows quadratically with $K$; more expensive for many classes.

In [ ]:
# One-vs-One
model_ovo = OneVsOneClassifier(
    LogisticRegression(random_state=42, max_iter=1000)
)
model_ovo.fit(X_train_scaled, y_train)

y_pred_ovo = model_ovo.predict(X_test_scaled)
print(f'OvO Accuracy: {accuracy_score(y_test, y_pred_ovo):.4f}')
print(f'Number of binary classifiers: {len(model_ovo.estimators_)}')
print(f'  (K=3 classes -> 3*2/2 = 3 classifiers)')

## Softmax Regression (Multinomial Logistic Regression)

Instead of decomposing the problem, we can generalize logistic regression directly.
The **softmax function** maps a vector of $K$ raw scores into a probability distribution:

$$P(y = k \mid \mathbf{x}) = \frac{e^{\mathbf{w}_k^T \mathbf{x}}}{\sum_{j=1}^{K} e^{\mathbf{w}_j^T \mathbf{x}}}$$

Properties:
- All probabilities are in (0, 1) and sum to 1
- Generalizes the sigmoid: for $K=2$, softmax reduces to the sigmoid function
- Trained by minimizing the **cross-entropy loss** across all classes
- All classifiers are trained **jointly**, so their outputs are properly calibrated

In [ ]:
# Visualize the softmax function
def softmax(z):
    """Compute softmax for a 2D array (samples x classes)."""
    exp_z = np.exp(z - z.max(axis=1, keepdims=True))  # subtract max for numerical stability
    return exp_z / exp_z.sum(axis=1, keepdims=True)

# Example: 3 classes, vary the score for class 0 while keeping others fixed
z0_range = np.linspace(-5, 5, 200)
scores = np.column_stack([z0_range, np.ones(200) * 0.5, np.ones(200) * -0.5])
probs = softmax(scores)

fig, ax = plt.subplots(figsize=(10, 5))
for k, name in enumerate(['Class 0', 'Class 1', 'Class 2']):
    ax.plot(z0_range, probs[:, k], linewidth=2, label=name)

ax.set_xlabel('Score for Class 0 (z₀)', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Softmax: How Changing One Score Affects All Probabilities', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Multinomial logistic regression in sklearn
model_multi = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    random_state=42,
    max_iter=1000,
)
model_multi.fit(X_train_scaled, y_train)

y_pred_multi = model_multi.predict(X_test_scaled)
print(f'Multinomial Accuracy: {accuracy_score(y_test, y_pred_multi):.4f}')

In [ ]:
# Compare all three approaches
print('Comparison of Multiclass Strategies:')
print(f'  OvR:          {accuracy_score(y_test, y_pred_ovr):.4f}')
print(f'  OvO:          {accuracy_score(y_test, y_pred_ovo):.4f}')
print(f'  Multinomial:  {accuracy_score(y_test, y_pred_multi):.4f}')

# Cross-validation for a fairer comparison
X_scaled_all = StandardScaler().fit_transform(X_iris)

print('\n5-Fold Cross-Validation Accuracy:')
for name, mc in [('OvR', 'ovr'), ('Multinomial', 'multinomial')]:
    m = LogisticRegression(multi_class=mc, solver='lbfgs', random_state=42, max_iter=1000)
    scores = cross_val_score(m, X_scaled_all, y_iris, cv=5, scoring='accuracy')
    print(f'  {name:15s}: {scores.mean():.4f} +/- {scores.std():.4f}')

> For well-behaved datasets like Iris, all strategies perform similarly. The multinomial
> approach is generally preferred because it trains a single coherent model and the
> probabilities are properly calibrated across all classes.

In [ ]:
# Inspect predicted probabilities
probas = model_multi.predict_proba(X_test_scaled)

# Show predictions for first 8 test samples
prob_df = pd.DataFrame(probas, columns=target_names)
prob_df['predicted'] = [target_names[p] for p in y_pred_multi]
prob_df['actual'] = [target_names[a] for a in y_test]
prob_df['correct'] = prob_df['predicted'] == prob_df['actual']

print('Predicted probabilities (first 8 samples):')
prob_df.head(8).round(4)

## Multiclass Confusion Matrix and Classification Report

The confusion matrix generalizes naturally: for $K$ classes, it becomes a $K \times K$
matrix where entry $(i, j)$ counts samples with true label $i$ predicted as label $j$.
The diagonal contains the correct predictions.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_multi,
    display_labels=target_names,
    ax=ax,
    cmap='Blues',
)
ax.set_title('Multiclass Confusion Matrix (Iris)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# The classification_report gives per-class precision, recall, and F1
print(classification_report(y_test, y_pred_multi, target_names=target_names))

### Understanding the Averaging Strategies

The classification report includes several summary rows:

- **macro avg:** Unweighted mean of per-class metrics. Treats all classes equally,
  regardless of size. Good when all classes matter equally.
- **weighted avg:** Weighted mean by class support (number of samples). Accounts for
  class imbalance.
- **micro avg** (not shown by default): Aggregate TP, FP, FN globally, then compute
  metrics. Equivalent to accuracy for multiclass.

## Visualizing Decision Boundaries for 3+ Classes

To visualize decision boundaries, we need to project onto 2 dimensions. We will use
the two most discriminative features from the Iris dataset.

In [ ]:
# Use only petal length and petal width (features 2 and 3) — the most informative
X_2d = X_iris[:, 2:4]
feature_pair = [feature_names[2], feature_names[3]]

# Scale and split
scaler_2d = StandardScaler()
X_2d_scaled = scaler_2d.fit_transform(X_2d)

# Fit multinomial on 2D data
model_2d = LogisticRegression(
    multi_class='multinomial', solver='lbfgs', random_state=42, max_iter=1000
)
model_2d.fit(X_2d_scaled, y_iris)

print(f'Accuracy with 2 features: {model_2d.score(X_2d_scaled, y_iris):.4f}')

In [ ]:
def plot_multiclass_boundary(model, X, y, class_names, feature_labels, title, ax=None):
    """Plot decision boundaries for a multiclass 2D classifier."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 7))

    margin = 0.5
    x_min, x_max = X[:, 0].min() - margin, X[:, 0].max() + margin
    y_min, y_max = X[:, 1].min() - margin, X[:, 1].max() + margin
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 400),
        np.linspace(y_min, y_max, 400),
    )

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    cmap = plt.cm.Set1
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap, levels=np.arange(-0.5, len(class_names), 1))
    ax.contour(xx, yy, Z, colors='black', linewidths=0.5, alpha=0.5)

    colors = [cmap(i / len(class_names)) for i in range(len(class_names))]
    for cls in range(len(class_names)):
        mask = y == cls
        ax.scatter(
            X[mask, 0], X[mask, 1],
            c=[colors[cls]], edgecolors='black', s=50,
            label=class_names[cls], zorder=3,
        )

    ax.set_xlabel(feature_labels[0])
    ax.set_ylabel(feature_labels[1])
    ax.set_title(title)
    ax.legend()

    return ax

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
plot_multiclass_boundary(
    model_2d, X_2d_scaled, y_iris,
    class_names=target_names,
    feature_labels=['Petal Length (scaled)', 'Petal Width (scaled)'],
    title='Multinomial Logistic Regression — Iris Decision Boundaries',
    ax=ax,
)
plt.tight_layout()
plt.show()

> Each region is where one class has the highest predicted probability. The boundaries
> between regions are **linear** — logistic regression always produces linear decision
> boundaries. Notice how setosa is perfectly separated, but versicolor and virginica
> overlap slightly.

### Comparing OvR vs Multinomial Boundaries

In [ ]:
model_2d_ovr = LogisticRegression(
    multi_class='ovr', solver='lbfgs', random_state=42, max_iter=1000
)
model_2d_ovr.fit(X_2d_scaled, y_iris)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, m, title in zip(
    axes,
    [model_2d_ovr, model_2d],
    ['One-vs-Rest (OvR)', 'Multinomial (Softmax)'],
):
    plot_multiclass_boundary(
        m, X_2d_scaled, y_iris,
        class_names=target_names,
        feature_labels=['Petal Length (scaled)', 'Petal Width (scaled)'],
        title=title,
        ax=ax,
    )

plt.tight_layout()
plt.show()

### Probability Surface

Let us also visualize the probability surface for each class.

In [ ]:
margin = 0.5
x_min, x_max = X_2d_scaled[:, 0].min() - margin, X_2d_scaled[:, 0].max() + margin
y_min, y_max = X_2d_scaled[:, 1].min() - margin, X_2d_scaled[:, 1].max() + margin
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300),
)

Z_proba = model_2d.predict_proba(np.c_[xx.ravel(), yy.ravel()])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for k, (ax, name) in enumerate(zip(axes, target_names)):
    proba_k = Z_proba[:, k].reshape(xx.shape)
    contour = ax.contourf(xx, yy, proba_k, levels=20, cmap='viridis', alpha=0.8)
    ax.contour(xx, yy, proba_k, levels=[0.5], colors='white', linewidths=2)

    for cls in range(3):
        mask = y_iris == cls
        ax.scatter(
            X_2d_scaled[mask, 0], X_2d_scaled[mask, 1],
            c='white' if cls == k else 'black',
            edgecolors='black', s=20, alpha=0.5,
        )

    ax.set_title(f'P({name})')
    ax.set_xlabel('Petal Length (scaled)')
    ax.set_ylabel('Petal Width (scaled)')
    plt.colorbar(contour, ax=ax)

plt.suptitle('Per-Class Probability Surfaces', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

> The softmax ensures that at every point, the three probabilities sum to exactly 1.
> Where one class has high probability, the others must have low probability.

## When to Use Logistic Regression vs Other Classifiers

Logistic regression is a powerful baseline, but it is not always the best choice.
Here is a practical guide:

| Scenario | Logistic Regression? | Better alternative |
|---|---|---|
| **Linearly separable data** | Excellent choice | — |
| **Need probability estimates** | Yes — well-calibrated by design | — |
| **Need interpretability** | Yes — coefficients are log-odds ratios | — |
| **Small dataset** | Good — few parameters, low risk of overfitting | — |
| **Non-linear decision boundary** | Poor | Decision trees, SVM with RBF kernel, neural networks |
| **Complex interactions** | Poor (unless you engineer features) | Random forest, gradient boosting |
| **Very high dimensional (text, genomics)** | Good with L1 regularization | — |
| **Large-scale dataset** | Good with `solver='saga'` | — |
| **Imbalanced classes** | Moderate (use class_weight) | Gradient boosting, ensemble methods |

**General advice:**
1. Always start with logistic regression as a **baseline** — it is fast, interpretable, and often surprisingly competitive
2. Move to more complex models only if the baseline clearly underperforms
3. If logistic regression works well, there is no reason to use a more complex model

### Quick Demonstration: Linear vs Non-Linear Data

In [ ]:
from sklearn.datasets import make_moons

# Linear data
X_lin, y_lin = make_classification(
    n_samples=300, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, random_state=42,
)

# Non-linear data (moons)
X_moon, y_moon = make_moons(n_samples=300, noise=0.2, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

datasets = [(X_lin, y_lin, 'Linearly Separable'), (X_moon, y_moon, 'Non-Linear (Moons)')]

for ax, (X_d, y_d, title) in zip(axes, datasets):
    scaler_d = StandardScaler()
    X_d_scaled = scaler_d.fit_transform(X_d)

    m = LogisticRegression(random_state=42, max_iter=1000)
    m.fit(X_d_scaled, y_d)
    acc = m.score(X_d_scaled, y_d)

    # Plot boundary
    margin = 0.5
    x_min, x_max = X_d_scaled[:, 0].min() - margin, X_d_scaled[:, 0].max() + margin
    y_min, y_max = X_d_scaled[:, 1].min() - margin, X_d_scaled[:, 1].max() + margin
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300),
    )
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    ax.scatter(X_d_scaled[y_d == 0, 0], X_d_scaled[y_d == 0, 1], c='blue', edgecolors='black', s=30)
    ax.scatter(X_d_scaled[y_d == 1, 0], X_d_scaled[y_d == 1, 1], c='red', edgecolors='black', s=30)
    ax.set_title(f'{title}\nLogistic Regression Accuracy: {acc:.3f}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

> Logistic regression excels on the linearly separable data but struggles with the
> moons dataset because the true boundary is curved. This is the fundamental limitation
> of any linear classifier.

## Key Takeaways

| Concept | Detail |
|---|---|
| **One-vs-Rest (OvR)** | Train $K$ binary classifiers; simple, scales linearly |
| **One-vs-One (OvO)** | Train $K(K-1)/2$ pairwise classifiers; voting-based prediction |
| **Softmax / Multinomial** | Direct generalization; jointly trained, properly calibrated probabilities |
| **sklearn default** | `multi_class='auto'` selects multinomial when solver supports it |
| **Multiclass confusion matrix** | $K \times K$ matrix; diagonal = correct predictions |
| **Averaging strategies** | Macro (equal class weight), weighted (by support), micro (global) |
| **Linear limitation** | Logistic regression produces linear boundaries; fails on non-linear patterns |
| **When to use** | Baseline for any classification task; interpretable, fast, well-calibrated |

---

**Chapter summary:** Logistic regression is a fundamental classification algorithm that
every data scientist should master. It provides interpretable coefficients, well-calibrated
probabilities, and a solid baseline for any classification problem. Understanding its
mechanics — the sigmoid, log-odds, MLE, evaluation metrics, and multiclass extensions —
lays the groundwork for understanding more complex classifiers.